
# 🎬 CineMind — AI Movie Discovery & Recommendation System


In [1]:

# Install once if required:
# !pip install pandas numpy scikit-learn sentence-transformers matplotlib seaborn

import ast
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 150)


In [2]:

# Update these paths for your machine
MOVIES_PATH = r"C:\Users\user\Downloads\movie_recommendation\tmdb_5000_movies.csv"
CREDITS_PATH = r"C:\Users\user\Downloads\movie_recommendation\tmdb_5000_credits.csv"

movies = pd.read_csv(MOVIES_PATH)
credits = pd.read_csv(CREDITS_PATH)

print("Movies:", movies.shape)
print("Credits:", credits.shape)
movies.head()


Movies: (4803, 20)
Credits: (4803, 4)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 878, ""name"": ""Science Fiction""}]",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"": 2964, ""name"": ""future""}, {""id"": 3386, ""name"": ""space war""}, {""id"": 3388, ""name"": ""space colony""}, {...",en,Avatar,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and prot...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289}, {""name"": ""Twentieth Century Fox Film Corporation"", ""id"": 306}, {""name"": ""Dune Entertainment"", ""id...","[{""iso_3166_1"": ""US"", ""name"": ""United States of America""}, {""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""}]",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso_639_1"": ""es"", ""name"": ""Espa\u00f1ol""}]",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 28, ""name"": ""Action""}]",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""name"": ""drug abuse""}, {""id"": 911, ""name"": ""exotic island""}, {""id"": 1319, ""name"": ""east india trading c...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of the Earth with Will Turner and Elizabeth Swann. But...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""name"": ""Jerry Bruckheimer Films"", ""id"": 130}, {""name"": ""Second Mate Productions"", ""id"": 19936}]","[{""iso_3166_1"": ""US"", ""name"": ""United States of America""}]",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 80, ""name"": ""Crime""}]",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name"": ""based on novel""}, {""id"": 4289, ""name"": ""secret agent""}, {""id"": 9663, ""name"": ""sequel""}, {""id"": 1...",en,Spectre,A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. While M battles political forces to keep the secret se...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""name"": ""Danjaq"", ""id"": 10761}, {""name"": ""B24"", ""id"": 69434}]","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""}, {""iso_3166_1"": ""US"", ""name"": ""United States of America""}]",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""}, {""iso_639_1"": ""en"", ""name"": ""English""}, {""iso_639_1"": ""es"", ""name"": ""Espa\u00f1ol""}, {""iso_639_1"": ...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""name"": ""Crime""}, {""id"": 18, ""name"": ""Drama""}, {""id"": 53, ""name"": ""Thriller""}]",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853, ""name"": ""crime fighter""}, {""id"": 949, ""name"": ""terrorist""}, {""id"": 1308, ""name"": ""secret identity""}...",en,The Dark Knight Rises,"Following the death of District Attorney Harvey Dent, Batman assumes responsibility for Dent's crimes to protect the late attorney's reputation an...",112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""name"": ""Warner Bros."", ""id"": 6194}, {""name"": ""DC Entertainment"", ""id"": 9993}, {""name"": ""Syncopy"", ""i...",

In [3]:

# Merge using the TMDB movie ID where possible.
# The credits file commonly calls this column 'movie_id'.
credits = credits.rename(columns={"movie_id": "id"})

movies = movies.merge(
    credits[["id", "cast", "crew"]],
    on="id",
    how="left"
)

movies.shape


(4803, 22)

In [4]:

def parse_names(value, limit=None):
    """Extract names from TMDB JSON-like columns."""
    if pd.isna(value):
        return []

    try:
        data = ast.literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return []

    if not isinstance(data, list):
        return []

    names = []
    for item in data:
        if isinstance(item, dict) and item.get("name"):
            names.append(str(item["name"]))
            if limit and len(names) >= limit:
                break

    return names


def parse_director(value):
    """Extract the director from the crew column."""
    if pd.isna(value):
        return []

    try:
        data = ast.literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return []

    if not isinstance(data, list):
        return []

    return [
        str(x["name"])
        for x in data
        if isinstance(x, dict)
        and x.get("job") == "Director"
        and x.get("name")
    ][:1]


movies["genres_list"] = movies["genres"].apply(parse_names)
movies["keywords_list"] = movies["keywords"].apply(parse_names)
movies["cast_list"] = movies["cast"].apply(lambda x: parse_names(x, 5))
movies["director_list"] = movies["crew"].apply(parse_director)

movies[["title", "genres_list", "keywords_list", "cast_list", "director_list"]].head()


,title,genres_list,keywords_list,cast_list,director_list
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colony, society, space travel, futuristic, romance, space, alien, tribe, alien planet, cgi, marine, soldi...","[Sam Worthington, Zoe Saldana, Sigourney Weaver, Stephen Lang, Michelle Rodriguez]",[James Cameron]
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india trading company, love of one's life, traitor, shipwreck, strong woman, ship, alliance, calypso, afte...","[Johnny Depp, Orlando Bloom, Keira Knightley, Stellan Skarsgård, Chow Yun-fat]",[Gore Verbinski]
2,Spectre,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi6, british secret service, united kingdom]","[Daniel Craig, Christoph Waltz, Léa Seydoux, Ralph Fiennes, Monica Bellucci]",[Sam Mendes]
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret identity, burglar, hostage drama, time bomb, gotham city, vigilante, cover-up, superhero, villainess,...","[Christian Bale, Michael Caine, Gary Oldman, Anne Hathaway, Tom Hardy]",[Christopher Nolan]
4,John Carter,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel, princess, alien, steampunk, martian, escape, edgar rice burroughs, alien race, superhuman strength...","[Taylor Kitsch, Lynn Collins, Samantha Morton, Willem Dafoe, Thomas Haden Church]",[Andrew Stanton]


In [5]:

# Clean overview and metadata
movies["overview"] = movies["overview"].fillna("")
movies["title"] = movies["title"].fillna("Unknown")

# Normalize text for matching
def clean_tokens(items):
    cleaned = []
    for item in items:
        item = re.sub(r"[^a-zA-Z0-9]+", "", str(item).lower())
        if item:
            cleaned.append(item)
    return cleaned

movies["genres_tokens"] = movies["genres_list"].apply(clean_tokens)
movies["keywords_tokens"] = movies["keywords_list"].apply(clean_tokens)
movies["cast_tokens"] = movies["cast_list"].apply(clean_tokens)
movies["director_tokens"] = movies["director_list"].apply(clean_tokens)

# Weighted text representation.
# Repeating a field gives that field more influence in TF-IDF.
movies["tags"] = (
    movies["overview"].str.lower() + " " +
    movies["genres_tokens"].apply(lambda x: " ".join(x) + " ") +
    movies["genres_tokens"].apply(lambda x: " ".join(x) + " ") +
    movies["keywords_tokens"].apply(lambda x: " ".join(x) + " ") +
    movies["cast_tokens"].apply(lambda x: " ".join(x) + " ") +
    movies["director_tokens"].apply(lambda x: " ".join(x))
)

# Remove duplicate titles
movies = movies.drop_duplicates(subset="title").reset_index(drop=True)

print("Final movies:", movies.shape)
movies[["title", "tags"]].head()


Final movies: (4800, 31)


,title,tags
0,Avatar,"in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and prot..."
1,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, has come back to life and is headed to the edge of the earth with will turner and elizabeth swann. but..."
2,Spectre,a cryptic message from bond’s past sends him on a trail to uncover a sinister organization. while m battles political forces to keep the secret se...
3,The Dark Knight Rises,"following the death of district attorney harvey dent, batman assumes responsibility for dent's crimes to protect the late attorney's reputation an..."
4,John Carter,"john carter is a war-weary, former military captain who's inexplicably transported to the mysterious and exotic planet of barsoom (mars) and reluc..."


In [6]:

# -------------------------------
# 1. TF-IDF content representation
# -------------------------------

tfidf = TfidfVectorizer(
    max_features=15000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2
)

tfidf_matrix = tfidf.fit_transform(movies["tags"])

print("TF-IDF matrix:", tfidf_matrix.shape)


TF-IDF matrix: (4800, 15000)


In [8]:

# Content similarity
content_similarity = cosine_similarity(tfidf_matrix)

title_to_index = pd.Series(
    movies.index,
    index=movies["title"].str.lower()
).drop_duplicates()

def find_movie(title):
    """Find exact or close title matches."""
    title = str(title).strip().lower()

    if title in title_to_index:
        return int(title_to_index[title])

    matches = movies[
        movies["title"].str.lower().str.contains(re.escape(title), na=False)
    ]

    if len(matches):
        return int(matches.index[0])

    raise ValueError(f"Movie not found: {title}")


def normalize_series(s):
    s = pd.Series(s, index=movies.index, dtype=float)
    if s.max() == s.min():
        return pd.Series(0.5, index=movies.index)
    return (s - s.min()) / (s.max() - s.min())


# Popularity and rating signals
movies["rating_score"] = normalize_series(movies["vote_average"].fillna(0))

# Log makes extremely popular movies less dominant
movies["popularity_log"] = np.log1p(movies["popularity"].fillna(0))
movies["popularity_score"] = normalize_series(movies["popularity_log"])

# Confidence adjustment using vote count
movies["vote_confidence"] = np.log1p(movies["vote_count"].fillna(0))
movies["vote_confidence"] = normalize_series(movies["vote_confidence"])


In [9]:

# -------------------------------
# 2. Explainable Hybrid Recommender
# -------------------------------

def recommend(
    movie_title,
    n=10,
    similarity_weight=0.55,
    rating_weight=0.20,
    popularity_weight=0.10,
    confidence_weight=0.15,
    min_rating=0.0
):
    idx = find_movie(movie_title)

    content_scores = content_similarity[idx]

    final_score = (
        similarity_weight * content_scores +
        rating_weight * movies["rating_score"].values +
        popularity_weight * movies["popularity_score"].values +
        confidence_weight * movies["vote_confidence"].values
    )

    result = movies.copy()
    result["content_similarity"] = content_scores
    result["recommendation_score"] = final_score

    # Exclude selected movie
    result = result[result.index != idx]

    # Optional rating filter
    result = result[result["vote_average"].fillna(0) >= min_rating]

    result = result.sort_values(
        "recommendation_score",
        ascending=False
    ).head(n)

    selected = movies.iloc[idx]

    output = result[
        [
            "title",
            "vote_average",
            "popularity",
            "content_similarity",
            "recommendation_score"
        ]
    ].copy()

    # Human-readable explanation
    selected_genres = set(selected["genres_tokens"])
    selected_cast = set(selected["cast_tokens"])
    selected_director = set(selected["director_tokens"])

    explanations = []

    for movie_idx in result.index:
        row = movies.loc[movie_idx]

        genre_match = selected_genres.intersection(row["genres_tokens"])
        cast_match = selected_cast.intersection(row["cast_tokens"])
        director_match = selected_director.intersection(row["director_tokens"])

        reasons = []

        if genre_match:
            reasons.append("Genre: " + ", ".join(sorted(genre_match)[:3]))
        if cast_match:
            reasons.append("Cast: " + ", ".join(sorted(cast_match)[:2]))
        if director_match:
            reasons.append("Director: " + ", ".join(sorted(director_match)[:1]))

        if not reasons:
            reasons.append("Similar story/themes")

        explanations.append("; ".join(reasons))

    output["why_recommended"] = explanations

    return output.reset_index(drop=True)


recommend("Interstellar", n=10)


,title,vote_average,popularity,content_similarity,recommendation_score,why_recommended
0,Guardians of the Galaxy,7.9,481.098624,0.132646,0.466705,"Genre: adventure, sciencefiction"
1,The Martian,7.6,167.932870,0.153328,0.451994,"Genre: adventure, drama, sciencefiction; Cast: jessicachastain"
2,The Matrix,7.9,104.309993,0.121577,0.436757,Genre: sciencefiction
3,Gravity,7.3,110.153618,0.138589,0.428025,"Genre: drama, sciencefiction"
4,Contact,7.2,55.249434,0.188401,0.420068,"Genre: drama, sciencefiction; Cast: matthewmcconaughey"
5,Inception,8.1,167.583710,0.050487,0.415438,"Genre: adventure, sciencefiction; Director: christophernolan"
6,The Dark Knight,8.2,187.322927,0.043424,0.413045,Genre: drama; Cast: michaelcaine; Director: christophernolan
7,2001: A Space Odyssey,7.9,86.201184,0.112329,0.411749,"Genre: adventure, sciencefiction"
8,Ex Machina,7.6,95.130041,0.105214,0.410473,"Genre: drama, sciencefiction"
9,Mad Max: Fury Road,7.2,434.278564,0.045787,0.402908,"Genre: adventure, sciencefiction"


In [10]:

# -------------------------------
# 3. User Taste Profile
# -------------------------------

def build_user_profile(liked_movies):
    """Create a profile from multiple movies the user likes."""
    indices = [find_movie(title) for title in liked_movies]

    # Average TF-IDF vectors of liked movies
    profile_vector = tfidf_matrix[indices].mean(axis=0)

    # Convert to 1 x N sparse-compatible matrix
    profile_vector = np.asarray(profile_vector)

    similarity_to_profile = cosine_similarity(
        profile_vector,
        tfidf_matrix
    )[0]

    return indices, similarity_to_profile


def recommend_for_user(liked_movies, n=10, min_rating=0):
    indices, profile_similarity = build_user_profile(liked_movies)

    scores = (
        0.65 * profile_similarity +
        0.20 * movies["rating_score"].values +
        0.15 * movies["popularity_score"].values
    )

    result = movies.copy()
    result["profile_similarity"] = profile_similarity
    result["recommendation_score"] = scores

    result = result[
        ~result.index.isin(indices) &
        (result["vote_average"].fillna(0) >= min_rating)
    ]

    return result.sort_values(
        "recommendation_score",
        ascending=False
    ).head(n)[
        [
            "title",
            "vote_average",
            "popularity",
            "profile_similarity",
            "recommendation_score"
        ]
    ].reset_index(drop=True)


liked = [
    "Interstellar",
    "Inception",
    "The Dark Knight"
]

recommend_for_user(liked, n=10)


,title,vote_average,popularity,profile_similarity,recommendation_score
0,The Dark Knight Rises,7.6,112.312950,0.294985,0.448451
1,Batman Begins,7.5,115.040024,0.246059,0.415175
2,2001: A Space Odyssey,7.9,86.201184,0.206861,0.391372
3,Guardians of the Galaxy,7.9,481.098624,0.124155,0.375465
4,Minority Report,7.1,65.948959,0.181017,0.352723
5,Contact,7.2,55.249434,0.172811,0.345534
6,Gattaca,7.5,70.398356,0.151089,0.342694
7,Sherlock Holmes: A Game of Shadows,7.0,81.499621,0.159369,0.341275
8,Batman Returns,6.6,59.113174,0.177587,0.338109
9,Mad Max: Fury Road,7.2,434.278564,0.089221,0.336497


In [14]:

# -------------------------------
# 4. User Movie DNA
# -------------------------------

from collections import Counter

def movie_dna(liked_movies):
    indices = [find_movie(x) for x in liked_movies]

    genres = Counter()
    keywords = Counter()

    for idx in indices:
        genres.update(movies.loc[idx, "genres_list"])
        keywords.update(movies.loc[idx, "keywords_list"])

    top_genres = genres.most_common(8)
    top_keywords = keywords.most_common(12)

    return top_genres, top_keywords


top_genres, top_keywords = movie_dna(liked)

print("🎬 YOUR MOVIE DNA")
print("\nTop Genres:")
for genre, count in top_genres:
    print(f"{genre}: {count}")

print("\nTop Themes/Keywords:")
for keyword, count in top_keywords:
    print(f"{keyword}: {count}")


🎬 YOUR MOVIE DNA

Top Genres:
Adventure: 2
Drama: 2
Science Fiction: 2
Action: 2
Thriller: 2
Mystery: 1
Crime: 1

Top Themes/Keywords:
imax: 2
saving the world: 1
artificial intelligence: 1
father son relationship: 1
single parent: 1
nasa: 1
expedition: 1
wormhole: 1
space travel: 1
famine: 1
black hole: 1
dystopia: 1


In [15]:

# -------------------------------
# 5. Mood-based Recommendations
# -------------------------------

MOOD_PROFILES = {
    "happy": ["comedy", "family", "music", "fun", "friendship"],
    "emotional": ["drama", "love", "family", "relationship", "emotional"],
    "scared": ["horror", "thriller", "mystery", "dark", "psychological"],
    "romantic": ["romance", "love", "relationship", "wedding", "couple"],
    "mind-blown": ["sciencefiction", "mystery", "space", "time", "psychological"],
    "excited": ["action", "adventure", "fight", "war", "superhero"],
    "relaxed": ["comedy", "family", "friendship", "music", "feelgood"]
}

def mood_recommend(mood, n=10, min_rating=0):
    mood = mood.lower().strip()

    if mood not in MOOD_PROFILES:
        raise ValueError(
            f"Choose one of: {', '.join(MOOD_PROFILES.keys())}"
        )

    mood_text = " ".join(MOOD_PROFILES[mood])
    mood_vector = tfidf.transform([mood_text])

    mood_similarity = cosine_similarity(
        mood_vector,
        tfidf_matrix
    )[0]

    score = (
        0.60 * mood_similarity +
        0.25 * movies["rating_score"].values +
        0.15 * movies["popularity_score"].values
    )

    result = movies.copy()
    result["mood_similarity"] = mood_similarity
    result["recommendation_score"] = score

    result = result[
        result["vote_average"].fillna(0) >= min_rating
    ]

    return result.sort_values(
        "recommendation_score",
        ascending=False
    ).head(n)[
        [
            "title",
            "vote_average",
            "popularity",
            "mood_similarity",
            "recommendation_score"
        ]
    ].reset_index(drop=True)


mood_recommend("mind-blown", n=10)


,title,vote_average,popularity,mood_similarity,recommendation_score
0,Gattaca,7.5,70.398356,0.301351,0.462797
1,Inception,8.1,167.583710,0.231035,0.454626
2,2001: A Space Odyssey,7.9,86.201184,0.256822,0.450506
3,Minority Report,7.1,65.948959,0.273650,0.434752
4,Cube,6.9,44.656151,0.275739,0.422531
5,Interstellar,8.1,724.247784,0.117096,0.418562
6,Source Code,7.1,59.198880,0.245580,0.415557
7,Contact,7.2,55.249434,0.227602,0.405768
8,Signs,6.4,28.848187,0.274195,0.399696
9,Self/less,6.3,66.097437,0.244247,0.397159


In [16]:

# -------------------------------
# 6. Natural Language Search
# -------------------------------

# For semantic search we use Sentence Transformers.
# This understands meaning better than keyword matching.

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Build a compact text representation for semantic search
movies["semantic_text"] = (
    movies["title"].fillna("") + ". " +
    movies["overview"].fillna("") + ". " +
    movies["genres_list"].apply(lambda x: ", ".join(x)) + ". " +
    movies["keywords_list"].apply(lambda x: ", ".join(x))
)

movie_embeddings = embedding_model.encode(
    movies["semantic_text"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
)

movie_embeddings.shape


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\user\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/150 [00:00<?, ?it/s]

(4800, 384)

In [17]:

def natural_language_search(query, n=10, min_rating=0):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    semantic_scores = cosine_similarity(
        query_embedding,
        movie_embeddings
    )[0]

    # Hybrid ranking
    final_score = (
        0.70 * semantic_scores +
        0.20 * movies["rating_score"].values +
        0.10 * movies["popularity_score"].values
    )

    result = movies.copy()
    result["semantic_similarity"] = semantic_scores
    result["recommendation_score"] = final_score

    result = result[
        result["vote_average"].fillna(0) >= min_rating
    ]

    return result.sort_values(
        "recommendation_score",
        ascending=False
    ).head(n)[
        [
            "title",
            "vote_average",
            "popularity",
            "semantic_similarity",
            "recommendation_score"
        ]
    ].reset_index(drop=True)


query = "A science fiction movie about space exploration, survival and emotional relationships"
natural_language_search(query, n=10)


,title,vote_average,popularity,semantic_similarity,recommendation_score
0,Gravity,7.3,110.153618,0.584308,0.624539
1,Interstellar,8.1,724.247784,0.510533,0.616576
2,Star Trek Into Darkness,7.4,78.291018,0.516268,0.573926
3,Sunshine,7.0,51.502884,0.527868,0.567962
4,Star Trek Beyond,6.6,65.352913,0.517444,0.556120
5,The Martian,7.6,167.932870,0.461537,0.550777
6,Moonraker,5.9,29.887404,0.530956,0.540294
7,Melancholia,7.0,36.173598,0.488406,0.535243
8,Aliens,7.7,67.660940,0.449355,0.530962
9,Seeking a Friend for the End of the World,6.3,30.863082,0.504410,0.530171


In [18]:

# -------------------------------
# 7. Controllable Recommendations
# -------------------------------

def controllable_recommendation(
    liked_movie,
    more_genres=None,
    less_genres=None,
    n=10,
    min_rating=0
):
    more_genres = [x.lower().replace(" ", "") for x in (more_genres or [])]
    less_genres = [x.lower().replace(" ", "") for x in (less_genres or [])]

    idx = find_movie(liked_movie)

    base_similarity = content_similarity[idx]

    scores = 0.70 * base_similarity

    for i, row in movies.iterrows():
        row_genres = set(row["genres_tokens"])

        # Boost desired genres
        if set(more_genres).intersection(row_genres):
            scores[i] += 0.20

        # Penalize unwanted genres
        if set(less_genres).intersection(row_genres):
            scores[i] -= 0.15

    # Add rating/popularity
    scores += (
        0.07 * movies["rating_score"].values +
        0.03 * movies["popularity_score"].values
    )

    result = movies.copy()
    result["recommendation_score"] = scores

    result = result[
        (result.index != idx) &
        (result["vote_average"].fillna(0) >= min_rating)
    ]

    return result.sort_values(
        "recommendation_score",
        ascending=False
    ).head(n)[
        [
            "title",
            "vote_average",
            "recommendation_score"
        ]
    ].reset_index(drop=True)


controllable_recommendation(
    "Interstellar",
    more_genres=["romance"],
    less_genres=["horror"],
    n=10
)


,title,vote_average,recommendation_score
0,Gattaca,7.5,0.342051
1,Deep Impact,5.9,0.321801
2,The Prisoner of Zenda,8.4,0.319255
3,Flyboys,6.3,0.312002
4,"Crouching Tiger, Hidden Dragon",7.2,0.309763
5,Legends of the Fall,7.2,0.308111
6,Her,7.9,0.307734
7,2046,6.9,0.305004
8,House of Flying Daggers,7.1,0.304942
9,The Last of the Mohicans,7.1,0.304373


In [19]:

# -------------------------------
# 8. Similarity Explanation
# -------------------------------

def explain_recommendation(source_movie, recommended_movie):
    source_idx = find_movie(source_movie)
    rec_idx = find_movie(recommended_movie)

    source = movies.loc[source_idx]
    rec = movies.loc[rec_idx]

    common_genres = set(source["genres_tokens"]).intersection(
        rec["genres_tokens"]
    )

    common_cast = set(source["cast_tokens"]).intersection(
        rec["cast_tokens"]
    )

    common_director = set(source["director_tokens"]).intersection(
        rec["director_tokens"]
    )

    print(f"🎬 {recommended_movie}")
    print()
    print("Why recommended:")

    if common_genres:
        print("✓ Common genres:", ", ".join(common_genres))

    if common_cast:
        print("✓ Common cast:", ", ".join(common_cast))

    if common_director:
        print("✓ Same director:", ", ".join(common_director))

    print(
        f"✓ Content similarity: "
        f"{content_similarity[source_idx, rec_idx]:.2%}"
    )

    print(f"✓ Rating: {rec['vote_average']}/10")


explain_recommendation(
    "Interstellar",
    "The Martian"
)


🎬 The Martian

Why recommended:
✓ Common genres: adventure, sciencefiction, drama
✓ Common cast: jessicachastain
✓ Content similarity: 15.33%
✓ Rating: 7.6/10


In [20]:

# -------------------------------
# 9. Movie Exploration Path
# -------------------------------

def movie_journey(start_movie, steps=5):
    journey = [start_movie]
    current = start_movie

    for _ in range(steps - 1):
        recommendations = recommend(
            current,
            n=5,
            min_rating=6.0
        )

        next_movie = recommendations.iloc[0]["title"]

        if next_movie in journey:
            break

        journey.append(next_movie)
        current = next_movie

    return pd.DataFrame({
        "Step": range(1, len(journey) + 1),
        "Movie": journey
    })


movie_journey("Interstellar", steps=6)


,Step,Movie
0,1,Interstellar
1,2,Guardians of the Galaxy
2,3,Captain America: Civil War
3,4,Avengers: Age of Ultron
4,5,The Avengers


In [21]:

# -------------------------------
# 10. Simple Evaluation / Sanity Checks
# -------------------------------

def recommendation_quality_check(movie_title, n=10):
    recs = recommend(movie_title, n=n)

    print(f"Input movie: {movie_title}")
    print(f"Average recommended rating: {recs['vote_average'].mean():.2f}")
    print(
        "Average content similarity: "
        f"{recs['content_similarity'].mean():.2%}"
    )

    return recs

recommendation_quality_check("Interstellar")


Input movie: Interstellar
Average recommended rating: 7.69
Average content similarity: 10.92%


,title,vote_average,popularity,content_similarity,recommendation_score,why_recommended
0,Guardians of the Galaxy,7.9,481.098624,0.132646,0.466705,"Genre: adventure, sciencefiction"
1,The Martian,7.6,167.932870,0.153328,0.451994,"Genre: adventure, drama, sciencefiction; Cast: jessicachastain"
2,The Matrix,7.9,104.309993,0.121577,0.436757,Genre: sciencefiction
3,Gravity,7.3,110.153618,0.138589,0.428025,"Genre: drama, sciencefiction"
4,Contact,7.2,55.249434,0.188401,0.420068,"Genre: drama, sciencefiction; Cast: matthewmcconaughey"
5,Inception,8.1,167.583710,0.050487,0.415438,"Genre: adventure, sciencefiction; Director: christophernolan"
6,The Dark Knight,8.2,187.322927,0.043424,0.413045,Genre: drama; Cast: michaelcaine; Director: christophernolan
7,2001: A Space Odyssey,7.9,86.201184,0.112329,0.411749,"Genre: adventure, sciencefiction"
8,Ex Machina,7.6,95.130041,0.105214,0.410473,"Genre: drama, sciencefiction"
9,Mad Max: Fury Road,7.2,434.278564,0.045787,0.402908,"Genre: adventure, sciencefiction"
